In [ ]:
from pathlib import Path
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from itertools import chain
from tqdm import tqdm
from collections import defaultdict

import cv2
import yaml
import json


REPO_ROOT = Path.cwd().parent
CONFIG_PATH = REPO_ROOT / "configs" / "config.yaml"

In [ ]:
with open(CONFIG_PATH, 'r') as file:
    config = yaml.safe_load(file)

train_data_path = REPO_ROOT / config["data"]["train"]
test_data_path = REPO_ROOT / config["data"]["test"]

train_datasets = config["training_set"]
test_datasets = config["testing_set"]

train_image_paths = []
test_image_paths = []

for dataset in train_datasets:
    infrared_imgs = sorted((train_data_path / dataset / "infrared").glob("*.jpg"))
    visible_imgs = sorted((train_data_path / dataset / "visible").glob("*.jpg"))
    
    assert len(infrared_imgs) == len(visible_imgs), (
        f"Mismatch in {dataset}: "
        f"{len(infrared_imgs)} infrared vs {len(visible_imgs)} visible"
    )

    for infrared_image, visible_image in zip(infrared_imgs, visible_imgs):
        train_image_paths.append((infrared_image, visible_image))

for dataset in test_datasets:
    infrared_imgs = sorted((test_data_path / dataset / "infrared").glob("*.jpg"))
    visible_imgs = sorted((test_data_path / dataset / "visible").glob("*.jpg"))
    
    assert len(infrared_imgs) == len(visible_imgs), (
        f"Mismatch in {dataset}: "
        f"{len(infrared_imgs)} infrared vs {len(visible_imgs)} visible"
    )

    for infrared_image, visible_image in zip(infrared_imgs, visible_imgs):
        test_image_paths.append((infrared_image, visible_image))

In [ ]:
infrared_path, visible_path = test_image_paths[15392]

print(infrared_path)
print(visible_path)

In [ ]:
"""
Lasher dataset we have is ~210 GB and cannot all be 
loaded into the memory at once. Checking the pixel 
size to determine the number of images to dynamically 
load the images into memory while training
"""

"""
(576, 960): 1093184
(576, 768): 37820
(481, 631): 280838
(480, 630): 57726
"""


def get_image_shape(path):
    path = Path(path)
    with Image.open(path) as img:
        width, height = img.size
    return height, width


all_image_paths = chain.from_iterable(train_image_paths + test_image_paths)

all_image_paths = list(all_image_paths)

images_shapes = defaultdict(int)

with ThreadPoolExecutor(max_workers=144) as executor:
    for shape in tqdm(
        executor.map(get_image_shape, all_image_paths),
        total=len(all_image_paths),
    ):
        images_shapes[shape] += 1

print(images_shapes)

In [ ]:
for img_size, count in images_shapes.items():
    print(f"{img_size}: {count}")

In [ ]:
MANIFEST_DIR = REPO_ROOT / "manifests"
MANIFEST_DIR.mkdir(exist_ok=True)

def save_manifest(image_pairs, output_path):
    with open(output_path, "w") as f:
        for infrared_path, visible_path in image_pairs:
            item = {
                "infrared": str(infrared_path.relative_to(REPO_ROOT)),
                "visible": str(visible_path.relative_to(REPO_ROOT)),
            }
            f.write(json.dumps(item) + "\n")

save_manifest(train_image_paths, MANIFEST_DIR / "lasher_train.jsonl")
save_manifest(test_image_paths, MANIFEST_DIR / "lasher_test.jsonl")